## Clase 07: Limpieza y Transformación de Datos con Pandas

En la clase anterior aprendimos a cargar datos, explorarlos, filtrarlos y hacer operaciones vectorizadas básicas. Sin embargo, en el mundo real (y en cualquier área laboral), los datos nunca llegan perfectos. 

Hoy aprenderemos a identificar y solucionar problemas comunes en los datasets:
1. Valores duplicados.
2. Formatos de texto inconsistentes (espacios extra, errores ortográficos).
3. Tipos de datos incorrectos (números guardados como texto).
4. Manejo de valores nulos (missing values) y atípicos (outliers).
5. Estandarización de fechas.

In [ ]:
import pandas as pd
import numpy as np

# 1. Carga de datos
# Cargamos el archivo CSV que contiene la información de los empleados
df = pd.read_csv('empleados_original.csv')

# Veamos las primeras filas para identificar los problemas a simple vista
display(df.head(10))

# Revisemos los tipos de datos y los valores nulos [9]
df.info()

### 1. Tratamiento de Duplicados
Al observar los datos, o al consolidar bases de datos de diferentes departamentos, es común que se filtren registros duplicados. Pandas nos permite detectarlos y eliminarlos fácilmente con `.drop_duplicates()`

In [ ]:
# Contar cuántos registros duplicados exactos existen
duplicados = df.duplicated().sum()
print(f"Se encontraron {duplicados} filas duplicadas.")

# Eliminar duplicados manteniendo la primera aparición
df = df.drop_duplicates(keep='first')

# Reiniciar el índice para mantener el orden limpio [11]
df = df.reset_index(drop=True)
print("Duplicados eliminados exitosamente.")

### 2. Limpieza de Cadenas de Texto (Strings)
Los errores de digitación son el pan de cada día. Nombres con espacios en blanco al inicio o al final, y nombres de departamentos mal escritos o con abreviaturas (ej. `Vntas` en lugar de `Ventas`, o `Tecnologia` en lugar de `IT`) [5]. Utilizaremos operaciones vectorizadas `.str`

In [ ]:
# Eliminar espacios en blanco al inicio y al final de los nombres
df['nombre_completo'] = df['nombre_completo'].str.strip()

# Corregir los nombres de los departamentos usando un diccionario de reemplazo
correcciones_dept = {
    'Vntas': 'Ventas',
    'Tecnologia': 'IT'
}
# Limpiamos espacios primero y luego reemplazamos
df['departamento'] = df['departamento'].str.strip().replace(correcciones_dept)

# Verificamos los departamentos únicos resultantes
print("Departamentos estandarizados:", df['departamento'].unique())

### 3. Limpieza y Conversión de Datos Numéricos
La columna `salario` tiene un gran problema: contiene símbolos de moneda (`$`, `USD`), comas separadoras de miles, y textos como `"Desconocido"`. Pandas leyó esta columna como texto (Object), pero necesitamos que sea numérica (Float) para poder hacer cálculos, agregaciones o modelos predictivos.

In [ ]:
# Limpiamos los caracteres no deseados en el salario
df['salario'] = df['salario'].str.replace('$', '', regex=False) \
                             .str.replace('USD', '', regex=False) \
                             .str.replace(',', '', regex=False) \
                             .str.strip()

# Convertimos a numérico. 'coerce' forzará a que textos como "Desconocido" se conviertan en NaN (Nulo)
df['salario'] = pd.to_numeric(df['salario'], errors='coerce')

# Manejo de errores lógicos: Salarios negativos (ej. -50000) los convertimos a su valor absoluto
df['salario'] = df['salario'].abs()

print("Estadísticas del salario limpio:")
display(df['salario'].describe())

### 4. Manejo de Valores Nulos (NaN) y Valores Atípicos (Outliers)
En la columna `edad` notamos valores escritos en texto (`"veinticinco"`) y valores atípicos (un empleado con `999` años). 
Una técnica común de limpieza es reemplazar datos faltantes por una estadística de resumen (como la media o la mediana) cuando no queremos perder toda la fila.

In [ ]:
# 1. Corregimos el texto manual a número
df['edad'] = df['edad'].replace('veinticinco', '25')
df['edad'] = pd.to_numeric(df['edad'], errors='coerce')

# 2. Reemplazamos los outliers (edad 999) por NaN para no afectar nuestro promedio
df.loc[df['edad'] > 100, 'edad'] = np.nan

# 3. Imputamos (rellenamos) los valores nulos en edad con la mediana
mediana_edad = df['edad'].median()
df['edad'] = df['edad'].fillna(mediana_edad)

# Para la columna 'vehiculo_asignado', un nulo significa simplemente que no tiene vehículo
df['vehiculo_asignado'] = df['vehiculo_asignado'].fillna('No asignado')

display(df[['nombre_completo', 'edad', 'vehiculo_asignado']].head(7))

### 5. Estandarización de Fechas
Las fechas en la columna `fecha_ingreso` vienen en distintos formatos (`15/03/2021`, `2022-08-10`, `15-Mar-2022`). Pandas tiene una función sumamente potente llamada `to_datetime()` que intentará inferir e igualar todos los formatos bajo el estándar `YYYY-MM-DD`.

In [ ]:
# Convertimos la columna a formato fecha (datetime)
df['fecha_ingreso'] = pd.to_datetime(df['fecha_ingreso'], errors='coerce')

# Verificamos nuestro DataFrame completamente limpio
display(df)

# Exportamos el resultado a un nuevo CSV limpio, listo para Análisis Exploratorio (EDA) o Power BI
df.to_csv('empleados_limpio.csv', index=False)
print("\n¡Datos limpios exportados exitosamente a 'empleados_limpio.csv'!")